# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samanashfaq05/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!git clone https://github.com/samanashfaq05/flyrank-ml-internship.git
%cd /content/flyrank-ml-internship

import os
print(os.getcwd())
print(os.path.exists("data/raw/content_refresh_anonymized.csv"))

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 175, done.
remote: Counting objects: 100% (175/175), done.
remote: Compressing objects: 100% (123/123), done.
remote: Total 175 (delta 79), reused 104 (delta 36), pack-reused 0 (from 0)
Receiving objects: 100% (175/175), 1.86 MiB | 7.37 MiB/s, done.
Resolving deltas: 100% (79/79), done.
/content/flyrank-ml-internship
/content/flyrank-ml-internship
True


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule: Prioritize pages that had meaningful search visibility, have experienced a recent drop in impressions, and have a weaker average search position. The rule gives higher scores to pages with larger impression declines and weaker positions.

Reason codes: declining_visible for pages with a clear impression decline and enough previous visibility; declining_weak_position when the page also has a weaker search position; no_priority when the page does not meet the rule conditions.

In [2]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:")
print(df.columns.tolist())

Rows: 30000
Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [5]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Calculate recent impression change
df["impression_change_pct"] = np.where(
    df["impressions_prev_30d"] > 0,
    (df["impressions_last_30d"] - df["impressions_prev_30d"])
    / df["impressions_prev_30d"],
    np.nan
)

# Two signals:
# 1. Recent impressions have dropped by at least 20%
# 2. The page had at least 100 previous impressions
df["impression_decline"] = df["impression_change_pct"] <= -0.20
df["meaningful_visibility"] = df["impressions_prev_30d"] >= 100

# Position signal: position 10 or worse
df["weak_position"] = df["avg_position"] >= 10

# Reason codes
df["reason_code"] = np.select(
    [
        df["impression_decline"] & df["meaningful_visibility"] & df["weak_position"],
        df["impression_decline"] & df["meaningful_visibility"]
    ],
    [
        "declining_weak_position",
        "declining_visible"
    ],
    default="no_priority"
)

print("Rows:", len(df))
print("\nReason codes:")
print(df["reason_code"].value_counts())

print("\nSignal summary:")
print("Pages with >=20% impression decline:",
      df["impression_decline"].sum())

print("Pages with meaningful previous visibility:",
      df["meaningful_visibility"].sum())

print("Pages with weak position:",
      df["weak_position"].sum())

Rows: 30000

Reason codes:
reason_code
no_priority                18899
declining_weak_position     6337
declining_visible           4764
Name: count, dtype: int64

Signal summary:
Pages with >=20% impression decline: 16305
Pages with meaningful previous visibility: 18010
Pages with weak position: 15962


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Scoring approach: I rank pages using a simple transparent score based on recent impression decline, meaningful previous visibility, and weaker average search position. Pages with a larger decline and weaker position receive higher priority. The ranked results are saved as work/outputs/baseline_action_score.csv for review.

In [4]:
import os
import numpy as np

# Create output folder
os.makedirs("work/outputs", exist_ok=True)

# Create a transparent baseline score
# Larger impression declines get higher scores.
# Pages with weak position receive an additional boost.

decline_strength = (-df["impression_change_pct"]).clip(lower=0)

df["score"] = (
    decline_strength
    * np.where(df["meaningful_visibility"], 1.0, 0.0)
    * np.where(df["weak_position"], 2.0, 1.0)
)

# Recommended action
df["action"] = np.where(
    df["score"] > 0,
    "review_for_refresh",
    "no_action"
)

# Rank all pages
df = df.sort_values("score", ascending=False).reset_index(drop=True)
df["rank"] = np.arange(1, len(df) + 1)

# Save the ranked queue
output_path = "work/outputs/baseline_action_score.csv"
df.to_csv(output_path, index=False)

print("Ranked pages:", len(df))
print("Output saved to:", output_path)

print("\nTop 10:")
print(
    df[
        [
            "rank",
            "content_id",
            "score",
            "reason_code",
            "action",
            "impression_change_pct",
            "impressions_prev_30d",
            "impressions_last_30d",
            "avg_position"
        ]
    ].head(10).to_string(index=False)
)

Ranked pages: 30000
Output saved to: work/outputs/baseline_action_score.csv

Top 10:
 rank           content_id  score             reason_code             action  impression_change_pct  impressions_prev_30d  impressions_last_30d  avg_position
    1 content_6bc2ec5f6061    2.0 declining_weak_position review_for_refresh                   -1.0                   124                     0          23.0
    2 content_56a9ec9120b0    2.0 declining_weak_position review_for_refresh                   -1.0                   144                     0          62.7
    3 content_98b92ff8b967    2.0 declining_weak_position review_for_refresh                   -1.0                   114                     0          64.8
    4 content_dde0d7418332    2.0 declining_weak_position review_for_refresh                   -1.0                   107                     0          34.3
    5 content_f05beee7738a    2.0 declining_weak_position review_for_refresh                   -1.0                   107    

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Top-20 review: I will review the highest-ranked pages to check whether the rule produces sensible refresh recommendations. For each page, I will record the recommended action, reason code, confidence level, and what could make the recommendation wrong.

In [7]:
# Load the ranked queue created in Section 2
top20_df = pd.read_csv("work/outputs/baseline_action_score.csv")

# Select the top 20 pages
top20 = top20_df.head(20).copy()

# Add confidence note
top20["confidence_note"] = np.where(
    top20["impression_change_pct"] <= -0.50,
    "High confidence: large recent impression decline",
    "Medium confidence: moderate impression decline"
)

# Add a careful limitation
top20["what_would_make_it_wrong"] = (
    "Recent impression loss may be temporary, seasonal, "
    "or caused by factors outside the content itself."
)

print("TOP-20 REVIEW")
print("=" * 100)

print(
    top20[
        [
            "rank",
            "content_id",
            "action",
            "reason_code",
            "confidence_note",
            "what_would_make_it_wrong"
        ]
    ].to_string(index=False)
)

TOP-20 REVIEW
 rank           content_id             action             reason_code                                  confidence_note                                                                            what_would_make_it_wrong
    1 content_6bc2ec5f6061 review_for_refresh declining_weak_position High confidence: large recent impression decline Recent impression loss may be temporary, seasonal, or caused by factors outside the content itself.
    2 content_56a9ec9120b0 review_for_refresh declining_weak_position High confidence: large recent impression decline Recent impression loss may be temporary, seasonal, or caused by factors outside the content itself.
    3 content_98b92ff8b967 review_for_refresh declining_weak_position High confidence: large recent impression decline Recent impression loss may be temporary, seasonal, or caused by factors outside the content itself.
    4 content_dde0d7418332 review_for_refresh declining_weak_position High confidence: large recent impression

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks and leakage check: I will examine whether any of the top-ranked pages could be weak recommendations and check that the baseline uses only information available before the decision. I will also check that no product decision flags or future outcome information are being used.

In [10]:
# Weak picks + leakage check

print("WEAK PICK REVIEW")
print("=" * 80)

# Look for potentially questionable top picks
weak_picks = top20[
    (top20["impressions_prev_30d"] < 120) |
    (top20["avg_position"] < 10)
].copy()

print("\nPotentially weak or questionable picks:")
print(
    weak_picks[
        [
            "rank",
            "content_id",
            "score",
            "reason_code",
            "impressions_prev_30d",
            "impressions_last_30d",
            "avg_position"
        ]
    ].to_string(index=False)
)

# Precise leakage check
print("\nLEAKAGE CHECK")
print("=" * 80)

# Columns that the baseline scoring rule is allowed to use
baseline_inputs = [
    "impressions_prev_30d",
    "impressions_last_30d",
    "avg_position"
]

leakage_columns = [
    "trend_direction",
    "trend_pct"
]

print("Baseline scoring inputs:")
for col in baseline_inputs:
    print(f"- {col}")

print("\nPotential leakage columns:")
for col in leakage_columns:
    print(f"- {col}: NOT USED in scoring rule")

print("\nConclusion:")
print("The baseline score is based on recent observable performance.")
print("trend_direction and trend_pct are not used to calculate the score.")
print("No client names, URLs, product flags, or future outcome columns are used.")

WEAK PICK REVIEW

Potentially weak or questionable picks:
 rank           content_id  score             reason_code  impressions_prev_30d  impressions_last_30d  avg_position
    3 content_98b92ff8b967    2.0 declining_weak_position                   114                     0          64.8
    4 content_dde0d7418332    2.0 declining_weak_position                   107                     0          34.3
    5 content_f05beee7738a    2.0 declining_weak_position                   107                     0          52.4
    6 content_094110291f27    2.0 declining_weak_position                   112                     0          10.3
    8 content_914c7f86875f    2.0 declining_weak_position                   114                     0          19.7
   11 content_1d115db7d35c    2.0 declining_weak_position                   119                     0          20.7
   12 content_dc01e2cc035d    2.0 declining_weak_position                   112                     0          47.1
   13 content_

Final observation: The baseline produced a transparent ranked refresh queue using recent observable performance signals. The weak-pick review showed that some high-ranked pages had relatively low previous visibility, so they may not deserve the same priority as pages with larger amounts of traffic. This reinforces that the baseline should support human review rather than make automatic refresh decisions.

Leakage check: The baseline score did not use trend_direction or trend_pct. It used only recent impression performance and average position. No client-identifying information, product flags, or future outcome information was used.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.